# 06b · LSTM + JEV sensor evidence pilot

06a의 동일한 fault3 test sequence 8개, 80행을 재사용하는 **post-hoc feature hypothesis** 실험이다. JEV는 LSTM의 5,000초 경보만 다시 평가한다. 06a의 첫 보수적 prompt, 세 판단 단계, JEV native score 임계값 1.0은 그대로 두고 sensor/state 숫자 2개만 추가한다. LSTM/RF를 재학습하거나 raw PHM 파일·checkpoint를 열지 않는다. 저장된 04b sequence artifact의 `x`는 당시 모델 입력 값이며 물리 단위로 해석하지 않는다.

In [ ]:
from pathlib import Path
import hashlib, json, math, os, time, urllib.error, urllib.request
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path(r"C:\projects\FabJudge-PHM2018-starter\FabJudge-PHM2018")
PRED = ROOT / "artifacts/huang2018/fault3_test_predictions.parquet"
SPLIT = ROOT / "artifacts/huang2018/split_metadata.json"
PILOT = ROOT / "artifacts/06a_lstm_jev/larger_test_01/pilot_rows.csv"
PILOT_SUMMARY = ROOT / "artifacts/06a_lstm_jev/larger_test_01/summary.json"
SEQUENCE_META = ROOT / "artifacts/huang2018/sequence_metadata.csv"
CONFIG = ROOT / "artifacts/huang2018/reproduction_config.json"
QUESTION_PATH = ROOT / "artifacts/06a_lstm_jev/prompt_tune_01/decision_question.json"
OUT = ROOT / "artifacts/06b_lstm_jev_sensor_skip"
CACHE = OUT / "cache"
MODEL = "typesafe/jev-1.13"
URL = "https://openrouter.ai/api/alpha/decisions"
TIMEOUT_SECONDS = 60
MAX_NEW_CALLS = 23
THRESHOLD_SECONDS = 5000

saved = pd.read_csv(PILOT)
predictions = pd.read_parquet(PRED)
metadata = json.loads(SPLIT.read_text(encoding="utf-8"))
original = json.loads(PILOT_SUMMARY.read_text(encoding="utf-8"))
test_ids = set(metadata["splits"]["fault3"]["test_original_sequences"])
prediction_ids = set(predictions.sequence_id.unique())
metadata_only, prediction_only = sorted(test_ids - prediction_ids), sorted(prediction_ids - test_ids)
if metadata_only or prediction_only:
    print({"metadata_only": metadata_only, "prediction_only": prediction_only})
    raise ValueError("Fault3 test split mismatch; JEV calls stopped")
if len(saved) != 80 or saved.sequence_id.nunique() != 8 or saved.duplicated(
        ["sequence_id", "raw_row_index"]).any():
    raise ValueError("06a 80-row pilot keys are not exact")
required = {"sequence_id", "raw_row_index", "wall_ttf_seconds", "lstm_pred_seconds",
            "rf_probability", "pipeline_rul_seconds"}
if required - set(predictions):
    raise ValueError(f"Missing prediction columns: {sorted(required - set(predictions))}")
work = saved[["sequence_id", "raw_row_index", "sample_position", "recent_lstm_delta",
              "recent_lstm_std", "history_count"]].merge(
    predictions, on=["sequence_id", "raw_row_index"], how="left", validate="one_to_one", sort=False)
if len(work) != 80 or work[["lstm_pred_seconds", "rf_probability", "wall_ttf_seconds"]].isna().any().any():
    raise ValueError("Prediction join is incomplete")
for name in ("lstm_pred_seconds", "rf_probability"):
    if not np.allclose(work[name], saved[name], rtol=0, atol=1e-9):
        raise ValueError(f"06a saved {name} differs from prediction parquet")
work["actual_fail_5000"] = work.wall_ttf_seconds <= THRESHOLD_SECONDS
work["lstm_alarm"] = work.lstm_pred_seconds <= THRESHOLD_SECONDS
work["pipeline_alarm"] = work.pipeline_rul_seconds <= THRESHOLD_SECONDS
if not (work.actual_fail_5000.to_numpy() == saved.actual_fail_5000.to_numpy()).all():
    raise ValueError("06a saved actual labels differ")
y, alarm = work.actual_fail_5000, work.lstm_alarm
baseline = {"TP": int((y & alarm).sum()), "FP": int((~y & alarm).sum()),
            "FN": int((y & ~alarm).sum()), "TN": int((~y & ~alarm).sum())}
expected = {"TP": 15, "FP": 8, "FN": 1, "TN": 56}
if baseline != expected or int(alarm.sum()) != 23 or any(original["metrics"][0][k] != v for k,v in expected.items()):
    raise ValueError(f"06a LSTM baseline mismatch: {baseline}; alarms={int(alarm.sum())}")
print({"split_exact_match": True, "rows": len(work), "sequences": len(test_ids),
       "lstm_alarm_rows": int(alarm.sum()), "LSTM": baseline})

## 저장된 sensor 입력과 후보

04b의 RF는 21개 `model_input_columns`와 residual 한 개를 사용했다. RF 모델을 읽지 않으므로 residual은 재계산하지 않는다. 저장된 RF sample은 80행과 exact key overlap이 없고, 별도 engineered CSV에는 동일 `sequence_id`·`raw_row_index` 키가 없다. 따라서 저장된 04b sequence `.npz`의 **현재 행과 과거 행만** 읽는다. 후보는 기존 fault3 중요도 목록에 등장한 6개 sensor의 현재값과, 기존 02 feature 정의를 최근 100개 관측에 적용한 6개 요약으로 제한한다. `samples_to_fault`, 고장 시각, 미래 행은 읽지 않는다.

In [ ]:
config = json.loads(CONFIG.read_text(encoding="utf-8"))
input_columns = config["model_input_columns"]
sequence_paths = pd.read_csv(SEQUENCE_META, usecols=["sequence_id", "sequence_path"])
if sequence_paths.sequence_id.duplicated().any():
    raise ValueError("Non-unique sequence artifact mapping")
work = work.merge(sequence_paths, on="sequence_id", how="left", validate="many_to_one", sort=False)
if work.sequence_path.isna().any():
    raise ValueError("Missing saved sequence artifact")
POINT_SENSORS = ["ROTATIONSPEED", "IONGAUGEPRESSURE", "ETCHSUPPRESSORCURRENT",
                 "FLOWCOOLPRESSURE", "ETCHBEAMCURRENT", "FLOWCOOLFLOWRATE"]
SUMMARY_SPECS = [("ROTATIONSPEED", "rms"), ("IONGAUGEPRESSURE", "shape_factor"),
                 ("ETCHSUPPRESSORCURRENT", "peak_abs"),
                 ("FLOWCOOLPRESSURE", "shape_factor"), ("FLOWCOOLPRESSURE", "peak_abs"),
                 ("FLOWCOOLFLOWRATE", "peak_abs")]
if not set(POINT_SENSORS).issubset(input_columns):
    raise ValueError("Candidate sensors missing from 04b input schema")
for sid, group in work.groupby("sequence_id", sort=False):
    path = (ROOT / str(group.sequence_path.iloc[0]).replace("\\", "/")).resolve()
    if not path.is_relative_to((ROOT / "artifacts/huang2018/sequences").resolve()):
        raise ValueError("Sequence artifact path outside expected directory")
    with np.load(path) as artifact:
        source_rows = artifact["raw_row_index"]
        x = artifact["x"]  # Never access artifact['samples_to_fault'].
        if x.shape[1] != len(input_columns) or not np.all(np.diff(source_rows) > 0):
            raise ValueError("Unexpected sequence input layout")
        positions = np.searchsorted(source_rows, group.raw_row_index.to_numpy())
        if np.any(positions >= len(source_rows)) or not np.array_equal(
                source_rows[positions], group.raw_row_index.to_numpy()):
            raise ValueError("Pilot row has no exact source measurement")
        for row_index, current in zip(group.index, positions):
            if current < 99:
                raise ValueError("100 past/current observations unavailable")
            trailing = x[current - 99:current + 1]
            for sensor in POINT_SENSORS:
                work.at[row_index, sensor] = float(x[current, input_columns.index(sensor)])
            for sensor, statistic in SUMMARY_SPECS:
                values = trailing[:, input_columns.index(sensor)].astype(float)
                rms = float(np.sqrt(np.mean(values ** 2)))
                if statistic == "rms": value = rms
                elif statistic == "peak_abs": value = float(np.max(np.abs(values)))
                else:
                    mean_abs = float(np.mean(np.abs(values)))
                    value = rms / mean_abs if mean_abs > 0 else np.nan
                work.at[row_index, f"{sensor}__{statistic}"] = value
CANDIDATES = POINT_SENSORS + [f"{s}__{stat}" for s,stat in SUMMARY_SPECS]
if not np.isfinite(work[CANDIDATES].to_numpy(float)).all():
    raise ValueError("Non-finite sensor candidate value")
print({"exact_artifact_joins": len(work), "candidate_features": len(CANDIDATES),
       "window_definition": "current and preceding 99 stored observations"})

## TP/FP 분리와 2개 feature 선정

효과 크기는 `(TP median − FP median) / sqrt((TP IQR² + FP IQR²)/2)`로 정의한다. 이 값은 작은 표본의 **탐색적 robust standardized gap**이며 유의성 검정이 아니다. 기존 fault3 중요도에서 상위권이고, 현재 23개 경보에서 다른 후보보다 상대적으로 분리되며 서로 상관이 거의 1이 아닌 두 `shape_factor`만 보낸다. 이 선정 자체가 test label을 본 사후 가설 생성이다. 분리 크기가 두 feature 모두 0.4 미만이거나 상관 절댓값이 0.9 이상이면 JEV 호출을 중단한다.

In [ ]:
alarm_rows = work.loc[work.lstm_alarm]
if len(alarm_rows) != 23 or int(alarm_rows.actual_fail_5000.sum()) != 15:
    raise ValueError("Alarm TP/FP membership changed")
separation = []
prior = pd.read_csv(ROOT / "artifacts/random_forest/aggregate_feature_ranking.csv")
prior_rank = prior.set_index("feature")["rank_ttf_flowcool_leak_seconds"].to_dict()
for feature in CANDIDATES:
    tp = alarm_rows.loc[alarm_rows.actual_fail_5000, feature].to_numpy(float)
    fp = alarm_rows.loc[~alarm_rows.actual_fail_5000, feature].to_numpy(float)
    tq1, tmed, tq3 = np.quantile(tp, [.25, .5, .75])
    fq1, fmed, fq3 = np.quantile(fp, [.25, .5, .75])
    tiqr, fiqr = tq3 - tq1, fq3 - fq1
    pooled_iqr = math.sqrt((tiqr ** 2 + fiqr ** 2) / 2)
    effect = (tmed - fmed) / pooled_iqr if pooled_iqr > 0 else np.nan
    separation.append({"feature": feature, "source": "current_04b_x" if feature in POINT_SENSORS else "causal_100_observation_summary",
                       "tp_median": float(tmed), "fp_median": float(fmed),
                       "tp_iqr": float(tiqr), "fp_iqr": float(fiqr),
                       "tp_q1": float(tq1), "tp_q3": float(tq3),
                       "fp_q1": float(fq1), "fp_q3": float(fq3),
                       "effect_size": float(effect) if math.isfinite(effect) else None,
                       "prior_fault3_rank": int(prior_rank[feature]) if feature in prior_rank and pd.notna(prior_rank[feature]) else None})
feature_table = pd.DataFrame(separation)
SELECTED = ["IONGAUGEPRESSURE__shape_factor", "FLOWCOOLPRESSURE__shape_factor"]
feature_table["selected"] = feature_table.feature.isin(SELECTED)
selected_effects = feature_table.set_index("feature").loc[SELECTED, "effect_size"].abs()
selected_correlation = float(alarm_rows[SELECTED].corr().iloc[0,1])
SENSOR_EVIDENCE_AVAILABLE = bool(np.isfinite(selected_effects).all() and
    (selected_effects >= 0.4).all() and abs(selected_correlation) < 0.9)
OUT.mkdir(parents=True, exist_ok=True)
feature_table.to_csv(OUT / "feature_separation.csv", index=False)
work.drop(columns=["sequence_path"]).to_csv(OUT / "pilot_rows_with_sensor_state.csv", index=False)
display(feature_table.loc[feature_table.selected | feature_table.feature.isin(
    ["ROTATIONSPEED", "FLOWCOOLFLOWRATE__peak_abs"]),
    ["feature", "tp_median", "fp_median", "tp_iqr", "fp_iqr", "effect_size", "selected"]])
print({"selected_features": SELECTED, "selected_abs_effects": selected_effects.round(3).to_dict(),
       "selected_correlation": round(selected_correlation, 3),
       "sensor_evidence_available": SENSOR_EVIDENCE_AVAILABLE})

## JEV sensor-skip gate

06a `prompt_tune_01`의 Decisions question을 **그대로** 사용한다. 추가되는 값은 위 두 sensor summary뿐이다. 따라서 이번 비교는 prompt 수정이 아니라 입력 정보 추가에 관한 것이다. Cache key는 endpoint, model, question, 7개 numeric state, timeout을 포함하는 실제 request 전체의 canonical JSON hash다.

In [ ]:
QUESTION = json.loads(QUESTION_PATH.read_text(encoding="utf-8"))
if QUESTION.get("type") != "score" or len(QUESTION.get("criteria", [])) != 3:
    raise ValueError("Unexpected 06a conservative question")

def load_api_key():
    values = {}
    env_file = ROOT / ".env"
    if env_file.is_file():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                name, value = line.split("=", 1)
                name, value = name.strip(), value.strip().strip('"').strip("'")
                if name in {"JEV_API_KEY", "OPENROUTER_API_KEY"} and value:
                    values[name] = value
    return (os.environ.get("JEV_API_KEY") or values.get("JEV_API_KEY") or
            os.environ.get("OPENROUTER_API_KEY") or values.get("OPENROUTER_API_KEY"))

API_KEY = load_api_key()  # Never print or serialize.
CACHE.mkdir(parents=True, exist_ok=True)

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      separators=(",", ":"), allow_nan=False)

def request_spec(row):
    state = {"lstm_pred_seconds": float(row.lstm_pred_seconds),
             "rf_probability": float(row.rf_probability),
             "recent_lstm_delta": float(row.recent_lstm_delta),
             "recent_lstm_std": float(row.recent_lstm_std),
             "history_count": int(row.history_count)}
    state.update({feature: float(row[feature]) for feature in SELECTED})
    if len(state) != 7 or not all(math.isfinite(value) for value in state.values()):
        raise ValueError("Invalid numeric JEV state")
    return {"endpoint": URL, "method": "POST", "timeout_seconds": TIMEOUT_SECONDS,
            "body": {"model": MODEL, "state": {"features": state},
                     "questions": {"alarm_evidence": QUESTION}}}

for _, row in work.loc[work.lstm_alarm].iterrows():
    body = request_spec(row)["body"]
    assert set(body) == {"model", "state", "questions"}
    assert set(body["state"]["features"]) == {
        "lstm_pred_seconds", "rf_probability", "recent_lstm_delta", "recent_lstm_std",
        "history_count", *SELECTED}

def validate_response(raw):
    if not isinstance(raw, dict) or not str(raw.get("model", "")).startswith(MODEL):
        raise ValueError("Unexpected response model")
    if raw.get("provider") != "TypeSafe":
        raise ValueError("Unexpected response provider")
    answers = raw.get("answers")
    if not isinstance(answers, dict) or set(answers) != {"alarm_evidence"}:
        raise ValueError("Unexpected answers schema")
    answer = answers["alarm_evidence"]
    if not isinstance(answer, dict) or answer.get("type") != "score":
        raise ValueError("Unexpected response format")
    score, confidence = answer.get("score"), answer.get("confidence")
    if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v)
           for v in (score, confidence)) or not 0 <= score <= 2 or not 0 <= confidence <= 1:
        raise ValueError("Missing, non-numeric, or out-of-range native score/confidence")
    probs = answer.get("probabilities")
    if not isinstance(probs, dict) or set(probs) != {"0", "1", "2"} or any(
            isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) or not 0 <= v <= 1
            for v in probs.values()) or not .95 <= sum(probs.values()) <= 1.05:
        raise ValueError("Invalid probabilities")
    if answer.get("legend") != {str(i): criterion for i, criterion in enumerate(QUESTION["criteria"])}:
        raise ValueError("Unexpected criterion legend")
    return float(score)

new_calls, cache_hits = 0, 0

def call_jev(row):
    global new_calls, cache_hits
    spec = request_spec(row)
    path = CACHE / (hashlib.sha256(canonical(spec).encode("utf-8")).hexdigest() + ".json")
    if path.is_file():
        cache_hits += 1
        try:
            record = json.loads(path.read_text(encoding="utf-8"))
            if record.get("request_sha256") != path.stem:
                raise ValueError("Cache request hash mismatch")
            score = validate_response(record["response"])
            return {"status": "cached", "score": score, "response": record["response"],
                    "error": None, "cache_hit": True}
        except (KeyError, TypeError, ValueError) as exc:
            return {"status": "failed", "score": None, "response": None,
                    "error": f"Invalid cache: {type(exc).__name__}: {exc}", "cache_hit": True}
    if not API_KEY:
        return {"status": "failed", "score": None, "response": None,
                "error": "OpenRouter credential unavailable", "cache_hit": False}
    if new_calls >= MAX_NEW_CALLS:
        return {"status": "failed", "score": None, "response": None,
                "error": "New API call cap reached", "cache_hit": False}
    req = urllib.request.Request(URL, data=canonical(spec["body"]).encode("utf-8"),
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}, method="POST")
    new_calls += 1
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=TIMEOUT_SECONDS) as response:
            response_raw = json.loads(response.read().decode("utf-8"))
        raw = {name: response_raw.get(name) for name in ("model", "provider", "answers", "usage")}
        if API_KEY in canonical(raw):
            raise ValueError("Credential appeared in response; refusing to cache")
        score = validate_response(raw)
        status, error = "success", None
    except urllib.error.HTTPError as exc:
        raw, score, status, error = None, None, "failed", f"HTTP {exc.code}"
    except Exception as exc:
        raw, score, status = None, None, "failed"
        error = f"{type(exc).__name__}: {str(exc).replace(API_KEY, '[REDACTED]')}"
    if status == "success":
        path.write_text(json.dumps({"request_sha256": path.stem, "response": raw,
            "latency_seconds": time.perf_counter() - started}, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"status": status, "score": score, "response": raw,
            "error": error, "cache_hit": False}

work["jev_score"] = np.nan
work["jev_status"] = np.where(work.lstm_alarm, "not_attempted", "not_called")
work["jev_called"] = False
work["cache_hit"] = False
work["jev_error"] = None
work["jev_keep"] = pd.Series(pd.NA, index=work.index, dtype="boolean")
work["hybrid_alarm"] = pd.Series(pd.NA, index=work.index, dtype="boolean")
work.loc[~work.lstm_alarm, "hybrid_alarm"] = False
responses = {}
if SENSOR_EVIDENCE_AVAILABLE:
    for ordinal, index in enumerate(work.index[work.lstm_alarm]):
        before = new_calls
        result = call_jev(work.loc[index])
        responses[index] = result
        work.at[index, "jev_called"] = new_calls > before
        work.at[index, "cache_hit"] = result["cache_hit"]
        work.at[index, "jev_status"] = result["status"]
        work.at[index, "jev_error"] = result["error"]
        if result["score"] is not None:
            work.at[index, "jev_score"] = result["score"]
            keep = result["score"] >= 1.0
            work.at[index, "jev_keep"] = keep
            work.at[index, "hybrid_alarm"] = keep
        if ordinal == 0:
            raw = result["response"] or {}
            answer = (raw.get("answers") or {}).get("alarm_evidence") or {}
            sanity = {"model": raw.get("model"), "provider": raw.get("provider"),
                      "response_format": answer.get("type"), "native_score_present": "score" in answer,
                      "score_numeric": isinstance(answer.get("score"), (int, float)) and not isinstance(answer.get("score"), bool),
                      "schema_valid": result["score"] is not None, "error": result["error"]}
            print("First response sanity:", sanity)
            if result["score"] is None:
                print("First response invalid; remaining JEV calls stopped.")
                break
else:
    print("No sufficiently separable sensor evidence; JEV API calls skipped.")
assert new_calls <= MAX_NEW_CALLS

## 80행 비교와 결론

동일 80행에서 LSTM, 기존 RF→LSTM, sensor-skip gate를 비교한다. JEV 실패 또는 미시도 경보는 hybrid missing으로 보존해 전체 지표를 임의로 채우지 않는다. 제거된 FP/TP는 성공적으로 판단한 행에서만 세며, 비교 완료 여부를 따로 기록한다.

In [ ]:
def metrics(name, predicted):
    unresolved = int(pd.isna(predicted).sum())
    result = {"system": name, "evaluation_rows": 80, "unresolved_rows": unresolved}
    if unresolved:
        return {**result, "TP": None, "FP": None, "FN": None, "TN": None,
                "precision": None, "recall": None}
    p = pd.Series(predicted, index=work.index).astype(bool)
    y = work.actual_fail_5000.astype(bool)
    tp, fp = int((p & y).sum()), int((p & ~y).sum())
    fn, tn = int((~p & y).sum()), int((~p & ~y).sum())
    return {**result, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": tp / (tp + fp) if tp + fp else None,
            "recall": tp / (tp + fn) if tp + fn else None}

def reported_sum(records, field):
    if not records:
        return "unavailable"
    values = []
    for record in records:
        value = ((record.get("response") or {}).get("usage") or {}).get(field)
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            return "unavailable"
        values.append(float(value))
    return sum(values)

metric_records = [metrics("LSTM only", work.lstm_alarm),
                  metrics("RF to LSTM pipeline", work.pipeline_alarm),
                  metrics("LSTM + JEV sensor-skip", work.hybrid_alarm)]
comparison = pd.DataFrame(metric_records)
rejected = work.loc[work.lstm_alarm & work.jev_score.notna() & work.jev_keep.fillna(False).eq(False)]
responded = [record for record in responses.values() if record["response"] is not None]
new_responded = [record for record in responded if not record["cache_hit"]]
kept = int((work.lstm_alarm & work.jev_keep.fillna(False)).sum())
removed = len(rejected)
summary = {
    "design": "post-hoc feature hypothesis on the same eight fault3 test sequences and 80 rows",
    "sensor_evidence_available": SENSOR_EVIDENCE_AVAILABLE,
    "selected_features": SELECTED if SENSOR_EVIDENCE_AVAILABLE else [],
    "feature_effect_definition": "(TP median - FP median) / sqrt((TP IQR^2 + FP IQR^2)/2)",
    "selected_feature_correlation": selected_correlation,
    "source": "stored 04b sequence x and raw_row_index; trailing 100 includes current and past only",
    "question_source": str(QUESTION_PATH),
    "pilot_rows": len(work), "test_sequences": len(test_ids), "lstm_alarm_rows": int(work.lstm_alarm.sum()),
    "jev_target_rows": 23 if SENSOR_EVIDENCE_AVAILABLE else 0,
    "jev_kept_alarm_rows": kept, "jev_removed_alarm_rows": removed,
    "removed_false_positives": int((~rejected.actual_fail_5000).sum()),
    "removed_true_positives": int(rejected.actual_fail_5000.sum()),
    "new_api_calls": new_calls, "cache_hits": cache_hits,
    "jev_success_rows": int(work.jev_status.isin(["success", "cached"]).sum()),
    "jev_failed_rows": int(work.jev_status.eq("failed").sum()),
    "jev_not_attempted_rows": int(work.jev_status.eq("not_attempted").sum()),
    "api_usage": {"input_tokens": reported_sum(responded, "input_tokens"),
                  "output_tokens": reported_sum(responded, "output_tokens"),
                  "reported_cost_usd": reported_sum(responded, "cost"),
                  "new_calls_reported_cost_usd": 0.0 if new_calls == 0 else reported_sum(new_responded, "cost")},
    "metrics": metric_records,
    "hybrid_comparison_complete": bool(work.hybrid_alarm.notna().all()),
}
row_columns = ["sequence_id", "raw_row_index", "sample_position", "lstm_pred_seconds",
               "rf_probability", "recent_lstm_delta", "recent_lstm_std", "history_count",
               *SELECTED, "lstm_alarm", "pipeline_alarm", "actual_fail_5000",
               "jev_score", "jev_status", "jev_called", "cache_hit", "jev_keep",
               "hybrid_alarm", "jev_error"]
work.loc[work.lstm_alarm, row_columns].to_csv(OUT / "jev_results.csv", index=False)
comparison.to_csv(OUT / "metric_comparison.csv", index=False)
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2,
                                       allow_nan=False), encoding="utf-8")
display(comparison)
print({key: summary[key] for key in ("lstm_alarm_rows", "jev_kept_alarm_rows",
    "jev_removed_alarm_rows", "removed_false_positives", "removed_true_positives",
    "new_api_calls", "cache_hits", "jev_success_rows", "jev_failed_rows")})
print("Reported API cost USD:", summary["api_usage"]["reported_cost_usd"])
if summary["hybrid_comparison_complete"]:
    print("Post-hoc sensor-input pilot complete. It generates a feature hypothesis, not independent validation.")
else:
    print("Hybrid comparison incomplete: JEV results unresolved or sensor evidence unavailable.")

### 관찰

두 sensor summary는 TP/FP 사이에서 탐색적 분리 경향을 보였지만, **같은 보수적 prompt에서** JEV native score는 23건 모두 1.0 이상이었다. LSTM의 TP 15건은 모두 보존했고 FP 8건도 모두 유지했다. 이 추가 입력은 이 post-hoc pilot의 경보 판정을 바꾸지 못했다. 이것만으로 입력 부족이 원인인지, prompt가 주된 원인인지 구분할 수 없다. 같은 8개 test sequence의 label을 보고 feature를 선택했으므로 일반화 개선의 증거로 해석하지 않는다.